# Clasificación SVM y agrupamiento K-Means
## Breast Cancer Wisconsin Dataset

**Dataset:** Breast Cancer Wisconsin Diagnostic Dataset  
**Fuente:** scikit-learn developers  
**Liga:** https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html  
**Fecha de consulta:** 12 de septiembre de 2026

Este conjunto de datos contiene características numéricas obtenidas de imágenes digitalizadas de masas mamarias. Cada registro representa un tumor y la variable objetivo permite distinguir entre tumores malignos y benignos.

La variable objetivo `target` tiene dos categorías:

- `0`: maligno
- `1`: benigno

Las variables predictoras contienen medidas como radio, textura, perímetro, área, concavidad y puntos cóncavos, entre otras.

### Pregunta de análisis supervisado
¿Es posible predecir correctamente si un tumor es benigno o maligno a partir de sus características numéricas utilizando modelos SVM?

### Pregunta de análisis no supervisado
¿Existen agrupamientos naturales entre los tumores a partir de sus características numéricas, sin utilizar previamente la clasificación de benigno o maligno?

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    silhouette_score
)

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)

## 2. Carga del conjunto de datos

In [ ]:
cancer = load_breast_cancer()

X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name="target")

df = X.copy()
df["target"] = y

df.head()

## 3. Revisión inicial

Se revisa el número de registros y variables, los tipos de datos, los valores faltantes, los duplicados y la distribución de la variable objetivo.

In [ ]:
print("Dimensiones del dataset:", df.shape)

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores faltantes totales:")
print(df.isnull().sum().sum())

print("\nRegistros duplicados:")
print(df.duplicated().sum())

print("\nDistribución de la variable objetivo:")
print(df["target"].value_counts())

print("\nDistribución porcentual:")
print((df["target"].value_counts(normalize=True) * 100).round(2))

El dataset contiene **569 registros**, **30 variables predictoras numéricas** y una variable objetivo.  
No presenta valores faltantes, por lo que no es necesario realizar imputación. Las etiquetas se conservan para el análisis supervisado y para la comparación final con los clusters.

## 4. Separación de variables predictoras y variable objetivo

In [ ]:
X = df.drop(columns="target")
y = df["target"]

# Copia de las etiquetas reales para comparación posterior
y_real = y.copy()

print("Predictoras:", X.shape)
print("Objetivo:", y.shape)

## 5. Importancia del escalamiento

Tanto SVM como K-Means pueden verse afectados cuando las variables tienen escalas muy diferentes.

SVM utiliza distancias y productos internos para construir su frontera de decisión. Si una variable tiene valores mucho mayores que otra, puede influir de forma desproporcionada.

K-Means también depende de distancias euclidianas. Por ello, una variable con valores grandes podría dominar la formación de los clusters.

Se utilizará `StandardScaler` para transformar las variables a una escala comparable.

# Análisis supervisado con SVM

## 6. División de entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Se utiliza una división de 80 % para entrenamiento y 20 % para prueba.  
La semilla `random_state=42` permite reproducir los resultados y `stratify=y` conserva la proporción de clases.

La evaluación sobre el conjunto de prueba permite medir el desempeño con datos que el modelo no utilizó durante su entrenamiento.

## 7. SVM con kernel lineal

In [ ]:
svm_lineal = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear"))
])

svm_lineal.fit(X_train, y_train)
y_pred_lineal = svm_lineal.predict(X_test)

acc_lineal = accuracy_score(y_test, y_pred_lineal)
prec_lineal = precision_score(y_test, y_pred_lineal)
rec_lineal = recall_score(y_test, y_pred_lineal)
f1_lineal = f1_score(y_test, y_pred_lineal)

print("Accuracy:", round(acc_lineal, 4))
print("Precision:", round(prec_lineal, 4))
print("Recall:", round(rec_lineal, 4))
print("F1-score:", round(f1_lineal, 4))

## 8. SVM con kernel RBF

In [ ]:
svm_rbf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

svm_rbf.fit(X_train, y_train)
y_pred_rbf = svm_rbf.predict(X_test)

acc_rbf = accuracy_score(y_test, y_pred_rbf)
prec_rbf = precision_score(y_test, y_pred_rbf)
rec_rbf = recall_score(y_test, y_pred_rbf)
f1_rbf = f1_score(y_test, y_pred_rbf)

print("Accuracy:", round(acc_rbf, 4))
print("Precision:", round(prec_rbf, 4))
print("Recall:", round(rec_rbf, 4))
print("F1-score:", round(f1_rbf, 4))

## 9. Comparación de modelos SVM

In [ ]:
resultados_svm = pd.DataFrame({
    "Modelo SVM": ["Modelo 1", "Modelo 2"],
    "Kernel": ["Lineal", "RBF"],
    "Accuracy": [acc_lineal, acc_rbf],
    "Precision": [prec_lineal, prec_rbf],
    "Recall": [rec_lineal, rec_rbf],
    "F1-score": [f1_lineal, f1_rbf]
})

resultados_svm.round(4)

El modelo con mejor desempeño se identifica comparando accuracy, precision, recall y F1-score.  
En este dataset, el kernel RBF suele obtener un desempeño ligeramente superior al lineal.

## 10. Matriz de confusión del mejor modelo

In [ ]:
# Selección automática del mejor modelo según F1-score
if f1_rbf >= f1_lineal:
    mejor_modelo = svm_rbf
    mejor_pred = y_pred_rbf
    mejor_nombre = "SVM RBF"
else:
    mejor_modelo = svm_lineal
    mejor_pred = y_pred_lineal
    mejor_nombre = "SVM Lineal"

cm = confusion_matrix(y_test, mejor_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Maligno", "Benigno"]
)

disp.plot()
plt.title(f"Matriz de confusión - {mejor_nombre}")
plt.show()

print(cm)

La matriz de confusión permite observar no solo cuántas predicciones fueron correctas, sino también qué tipo de errores cometió el modelo.

Esto es importante porque la accuracy por sí sola no muestra si el modelo está confundiendo principalmente tumores malignos con benignos o viceversa. En un contexto médico, ambos errores no tienen necesariamente la misma importancia.

## 11. Ajuste de hiperparámetro C

In [ ]:
valores_C = [0.1, 1, 10]
resultados_C = []

for C in valores_C:
    modelo = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=C))
    ])

    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    resultados_C.append({
        "C": C,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-score": f1_score(y_test, pred)
    })

pd.DataFrame(resultados_C).round(4)

El parámetro `C` controla el equilibrio entre mantener un margen amplio y penalizar los errores de clasificación.

- Un valor bajo de `C` permite una frontera más flexible y tolera algunos errores.
- Un valor alto de `C` intenta clasificar correctamente más observaciones de entrenamiento y puede generar una frontera más compleja.
- Valores demasiado altos pueden aumentar el riesgo de sobreajuste.

# Análisis no supervisado con K-Means

## 12. Escalamiento de variables

In [ ]:
scaler_kmeans = StandardScaler()
X_scaled = scaler_kmeans.fit_transform(X)

print("Dimensiones después del escalamiento:", X_scaled.shape)

La variable objetivo no se utiliza para entrenar K-Means.  
Las etiquetas reales se conservan únicamente para realizar una comparación posterior.

## 13. Evaluación de valores de k

In [ ]:
inercias = []
silhouettes = []

valores_k = range(2, 7)

for k in valores_k:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    clusters_temp = kmeans.fit_predict(X_scaled)

    inercias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, clusters_temp))

evaluacion_k = pd.DataFrame({
    "k": list(valores_k),
    "Inercia": inercias,
    "Silhouette Score": silhouettes
})

evaluacion_k.round(4)

### Método del codo

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(list(valores_k), inercias, marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.show()

### Silhouette Score

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(list(valores_k), silhouettes, marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")
plt.show()

El método del codo permite observar a partir de qué valor de `k` la disminución de la inercia comienza a ser menor.

El Silhouette Score evalúa qué tan bien separadas están las observaciones entre clusters. Valores mayores indican una mejor separación.

En este dataset, `k = 2` suele presentar el Silhouette Score más alto y además tiene sentido considerando la estructura general de los datos, por lo que se seleccionan dos clusters.

## 14. K-Means final con k = 2

In [ ]:
kmeans_final = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

clusters = kmeans_final.fit_predict(X_scaled)

df_clusters = X.copy()
df_clusters["Cluster"] = clusters

df_clusters.head()

## 15. Tamaño de cada cluster

In [ ]:
tamano_clusters = (
    df_clusters["Cluster"]
    .value_counts()
    .sort_index()
    .rename("Tamaño")
)

tamano_clusters

## 16. Perfil promedio de los clusters

In [ ]:
variables_principales = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean concavity",
    "mean concave points"
]

resumen_clusters = (
    df_clusters.groupby("Cluster")[variables_principales]
    .mean()
    .round(3)
)

resumen_clusters["Tamaño"] = tamano_clusters

resumen_clusters

Los clusters se diferencian principalmente por características relacionadas con tamaño y forma.

Uno de los grupos tiende a concentrar observaciones con menor radio, perímetro, área, concavidad y puntos cóncavos. El otro presenta valores más altos en estas variables, lo que indica tumores más grandes y con formas más irregulares.

Esto permite interpretar los clusters como dos perfiles generales de observaciones, aunque todavía no se han utilizado las etiquetas reales para definirlos.

# PCA y visualización

## 17. Proyección de los datos en dos dimensiones

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Cluster": clusters
})

print("Varianza explicada por PC1 y PC2:")
print(pca.explained_variance_ratio_)
print("Varianza acumulada:", pca.explained_variance_ratio_.sum())

## 18. Visualización de clusters

In [ ]:
plt.figure(figsize=(8, 6))

for cluster in np.unique(clusters):
    plt.scatter(
        pca_df.loc[pca_df["Cluster"] == cluster, "PC1"],
        pca_df.loc[pca_df["Cluster"] == cluster, "PC2"],
        label=f"Cluster {cluster}",
        alpha=0.7
    )

plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Clusters de K-Means proyectados con PCA")
plt.legend()
plt.show()

## 19. Visualización de las clases reales

In [ ]:
pca_real = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Clase real": y_real.values
})

plt.figure(figsize=(8, 6))

for clase in np.unique(y_real):
    plt.scatter(
        pca_real.loc[pca_real["Clase real"] == clase, "PC1"],
        pca_real.loc[pca_real["Clase real"] == clase, "PC2"],
        label=cancer.target_names[clase],
        alpha=0.7
    )

plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Clases reales proyectadas con PCA")
plt.legend()
plt.show()

La segunda gráfica utiliza las etiquetas reales únicamente para facilitar la comparación visual.  
Estas etiquetas no participaron en el entrenamiento ni en el ajuste de K-Means.

## 20. Comparación entre clusters y clases reales

In [ ]:
comparacion = pd.crosstab(
    clusters,
    y_real,
    rownames=["Cluster"],
    colnames=["Clase real"]
)

comparacion.columns = ["Maligno", "Benigno"]

comparacion

La comparación permite observar si existe correspondencia parcial entre los grupos encontrados por K-Means y las clases reales.

En este dataset suele observarse que uno de los clusters concentra principalmente tumores benignos, mientras que el otro concentra una mayor proporción de tumores malignos. Sin embargo, la separación no es perfecta.

Esto es esperado porque K-Means no intenta reproducir las etiquetas originales. El algoritmo únicamente agrupa observaciones de acuerdo con su similitud geométrica, mientras que las clases conocidas representan una clasificación clínica.

# Conclusión

Los enfoques supervisado y no supervisado permitieron analizar el conjunto Breast Cancer Wisconsin desde perspectivas diferentes.

El modelo SVM fue más sencillo de evaluar en términos predictivos, ya que las etiquetas reales permitieron medir directamente el número de aciertos y errores. El kernel RBF obtuvo un desempeño muy alto y mostró que las características numéricas de los tumores contienen información suficiente para distinguir entre clases con gran precisión.

Las métricas accuracy, precision, recall y F1-score permitieron analizar el desempeño desde diferentes perspectivas. La matriz de confusión también resultó importante para identificar específicamente los tipos de errores cometidos.

K-Means permitió analizar los datos sin utilizar la variable objetivo y encontrar agrupamientos naturales. Los clusters se diferenciaron principalmente por variables relacionadas con el tamaño y la irregularidad de las masas. Al comparar posteriormente estos grupos con las clases reales se encontró una correspondencia importante, aunque no perfecta.

Una de las principales limitaciones es que ambos métodos son sensibles a la escala de las variables, por lo que el escalamiento fue necesario. K-Means también depende de la forma geométrica de los grupos y puede tener dificultades cuando la estructura de los datos es más compleja. PCA fue útil para visualizar los patrones, aunque al reducir 30 variables a dos componentes se pierde parte de la información original.

En problemas donde existen categorías conocidas y se desea predecir la clase de nuevas observaciones, utilizaría aprendizaje supervisado como SVM. En cambio, cuando no existen etiquetas y el objetivo es explorar patrones, segmentos o estructuras naturales en los datos, utilizaría técnicas no supervisadas como K-Means.